# 03 — Generate 10,000 handymen

A handyman is a user with `is_handyman = true` plus a row in `handyman_details`, so this notebook writes:

* **`synth_users`** (appended, ids `N_USERS+1..N_USERS+N_HANDYMEN`) — the handyman accounts: name, email, password, date of birth, phone, address, coordinates, `is_handyman = true`. Re-running replaces only the handyman rows.
* **`synth_handyman_details`** — the same columns as `maintops.handyman_details` (`user_id`, specialisations, skills, experience summary, `cv_path`, `cv_raw_text`, `has_car`, price).
  `completed_cases`, `rating_avg` and `rating_count` start at 0 and are filled in by `05_generate_incidents` from the actual incident history, so they always agree with it.
* **`synth_handymen_truth`** — generator ground truth that is **not** part of the app schema: hidden `quality` (drives ratings later), years of experience and the structured CV (`cv_json`) that `04_generate_cv_pdfs` renders. Keyed by `user_id`.

Specialisations are picked from the controlled taxonomy, skills from that specialisation's skill list, and the CV is consistent with both.

In [1]:
%pip install argon2-cffi

Note: you may need to restart the kernel to use updated packages.


In [2]:
try:
    dbutils.library.restartPython()
except NameError:  # running locally: no dbutils, nothing to restart
    pass

In [3]:
# Environment bootstrap: runs unchanged on Databricks and on a local machine.
# Locally it connects through Databricks Connect (serverless unless DATABRICKS_CLUSTER_ID is set)
# and provides `spark`, `display` and `list_volume_files`. Override the profile with DATABRICKS_CONFIG_PROFILE.
import os
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
LOCAL_SPARK_DEPENDENCIES = []   # pip packages the remote workers need (serverless Databricks Connect only)
if not IS_DATABRICKS:
    os.environ.setdefault("DATABRICKS_CONFIG_PROFILE", "george.sokolovsky2001@gmail.com")
    from databricks.connect import DatabricksSession

    _builder = DatabricksSession.builder
    if os.environ.get("DATABRICKS_CLUSTER_ID"):
        _builder = _builder.clusterId(os.environ["DATABRICKS_CLUSTER_ID"])
    else:
        _builder = _builder.serverless()
        if LOCAL_SPARK_DEPENDENCIES:
            from databricks.connect import DatabricksEnv
            _builder = _builder.withEnvironment(DatabricksEnv().withDependencies(LOCAL_SPARK_DEPENDENCIES))
    spark = _builder.getOrCreate()

    def display(obj, n=1000):
        from IPython.display import display as _display
        _display(obj.limit(n).toPandas() if hasattr(obj, "toPandas") else obj)

def list_volume_files(directory, suffix=""):
    """File names in a Unity Catalog Volume folder (the Volume is only mounted on Databricks, so go through the Files API locally)."""
    if IS_DATABRICKS:
        return [f for f in os.listdir(directory) if f.endswith(suffix)] if os.path.isdir(directory) else []
    from databricks.sdk import WorkspaceClient
    return [e.name for e in WorkspaceClient().files.list_directory_contents(directory) if e.name.endswith(suffix)]


In [4]:
%run ./00_config

In [5]:
# Cell 1 — Load the handyman addresses
import json
import numpy as np, pandas as pd
from argon2 import PasswordHasher

addr = (spark.table(T_ADDRESS_POOL).where("purpose = 'handyman'").orderBy("address_id").toPandas())
assert len(addr) == N_HANDYMEN, f"expected {N_HANDYMEN} handyman addresses, found {len(addr)} — re-run 01_real_addresses"

In [6]:
# Cell 2 — CV / profile builders
CURRENT_YEAR = 2026

def pick_specialisations(rng):
    primary = rng.choice(list(SPEC_WEIGHTS), p=list(SPEC_WEIGHTS.values()))
    n = rng.choice([1, 2, 3], p=[.55, .35, .10])
    specs = [primary]
    candidates = RELATED[primary] + [s for s in SPECIALISATIONS if s not in RELATED[primary]]
    for c in candidates:
        if len(specs) >= n: break
        if c not in specs and (c in RELATED[primary] or rng.random() < .1):
            specs.append(c)
    return specs

def pick_skills(rng, specs):
    skills = list(rng.choice(SKILLS[specs[0]], size=rng.integers(3, 6), replace=False))
    for s in specs[1:]:
        skills += list(rng.choice(SKILLS[s], size=rng.integers(1, 4), replace=False))
    return skills

BULLETS = ["Carried out {s} for private households and small businesses", "Diagnosed faults and performed {s} on site",
           "Responsible for {s} including documentation of completed work", "Advised customers on options and costs for {s}",
           "Trained apprentices in {s}", "Planned and scheduled jobs involving {s}"]
ROLES = ["Journeyman", "Journeyman", "Senior technician", "Foreman (Vorarbeiter)"]

def build_cv(rng, first, last, email, phone, address_line, home_city, specs, skills, years):
    primary = specs[0]
    start_year = CURRENT_YEAR - years
    n_jobs = 1 if years < 4 else 2 if years < 10 else 3 if years < 20 else 4
    cuts = sorted(rng.choice(np.arange(1, years), size=n_jobs - 1, replace=False)) if n_jobs > 1 else []
    bounds = [0] + [int(c) for c in cuts] + [years]
    jobs = []
    for j in range(n_jobs):                                             # oldest -> newest
        y0, y1 = start_year + bounds[j], start_year + bounds[j + 1]
        self_employed = (j == n_jobs - 1) and years >= 8 and rng.random() < .3
        word = rng.choice(EMPLOYER_WORDS[primary])
        employer = f"{last} {word} (self-employed)" if self_employed else f"{rng.choice(LAST_NAMES)} {word} {rng.choice(EMPLOYER_SUFFIXES)}"
        role = "Owner-operator" if self_employed else ROLES[min(j, len(ROLES) - 1)]
        bullets = [b.format(s=s) for b, s in zip(rng.choice(BULLETS, size=3, replace=False), rng.choice(skills, size=3, replace=len(skills) < 3))]
        jobs.append({"role": f"{role} - {TITLES[primary]}", "employer": employer,
                     "city": home_city if j == n_jobs - 1 else str(rng.choice(OTHER_CITIES)),
                     "from": y0, "to": "present" if j == n_jobs - 1 else y1, "bullets": bullets})
    jobs.reverse()                                                      # newest first

    certs_all = list(CERTIFICATIONS[primary]) + (list(CERTIFICATIONS[specs[1]][:2]) if len(specs) > 1 else [])
    n_certs = 1 + (years > 5) + (years > 15)
    certs = [{"name": c, "year": int(rng.integers(min(start_year + 2, CURRENT_YEAR), CURRENT_YEAR + 1))}
             for c in rng.choice(certs_all, size=n_certs, replace=False)]
    education = [{"name": str(rng.choice(EDUCATION_SCHOOL)), "year": start_year - 3},
                 {"name": f"Vocational training (Ausbildung) - {TITLES[primary]}", "year": start_year}]
    languages = ["German (native)"] + [f"{l} ({rng.choice(['fluent', 'good', 'basic'])})" for l in rng.choice(LANGUAGES, size=rng.integers(0, 3), replace=False)]

    master = any("Meisterbrief" in c["name"] for c in certs)
    summary = (f"{years} years of experience as {TITLES[primary].lower()}, specialising in "
               f"{' and '.join(s.replace('_', ' ') for s in specs)}. Focus on {skills[0]}, {skills[1]} and {skills[2]} "
               f"for private households and small businesses." + (" Certified master craftsman." if master else ""))
    return {"name": f"{first} {last}", "headline": TITLES[primary], "email": email, "phone": phone, "address": address_line,
            "summary": summary, "specialisations": specs, "skills": skills, "jobs": jobs, "certifications": certs,
            "education": education, "languages": languages,
            "extras": ["Driving licence class B"] * int(rng.random() < .95) + ["Own work van and tools"] * int(rng.random() < .8)}

def cv_to_text(cv):
    lines = [cv["name"], cv["headline"], f"{cv['email']} | {cv['phone']} | {cv['address']}", "", "PROFILE", cv["summary"], "", "SKILLS", ", ".join(cv["skills"]), "", "EXPERIENCE"]
    for j in cv["jobs"]:
        lines += [f"{j['from']} - {j['to']}  {j['role']}, {j['employer']}, {j['city']}"] + [f"- {b}" for b in j["bullets"]]
    lines += ["", "CERTIFICATIONS"] + [f"{c['year']}  {c['name']}" for c in cv["certifications"]]
    lines += ["", "EDUCATION"] + [f"{e['year']}  {e['name']}" for e in cv["education"]]
    lines += ["", "LANGUAGES", ", ".join(cv["languages"])] + (["", "OTHER", ", ".join(cv["extras"])] if cv["extras"] else [])
    return "\n".join(lines)

In [7]:
# Cell 3 — Build the handymen
rng = np.random.default_rng(SEED + 2)
password_hash = PasswordHasher().hash(DEMO_PASSWORD)

# Handyman emails must not collide with any client email (everyone logs in with email).
# Drop handyman rows of an earlier run first so this notebook can be re-run on its own.
if spark.catalog.tableExists(T_USERS):
    spark.sql(f"DELETE FROM {T_USERS} WHERE is_handyman")
used_emails = {e.lower() for e in spark.table(T_USERS).select("email").toPandas()["email"]}
assert len(used_emails) == N_USERS, "run 02_generate_users first"

handy_users, details, truth = [], [], []
for idx in range(N_HANDYMEN):
    hid   = N_USERS + idx + 1                                            # users.id of this handyman
    a     = addr.iloc[idx]
    first, last = str(rng.choice(FIRST_NAMES)), str(rng.choice(LAST_NAMES))
    email = make_email(rng, first, last, used_emails)
    phone = make_phone(rng)
    house = f"{a.street} {a.housenumber}"[:128]
    years = int(np.clip(rng.gamma(2.5, 5.0), 1, 40))
    quality = float(np.clip(rng.beta(5, 2) + 0.004 * (years - 10), 0.05, 1.0))   # hidden; drives ratings in notebook 05
    specs  = pick_specialisations(rng)
    skills = pick_skills(rng, specs)
    cv     = build_cv(rng, first, last, email, phone, f"{house}, {a.postcode} {a.city}", a.city, specs, skills, years)
    price  = round(BASE_PRICE[specs[0]] * (0.8 + 0.5 * quality) * (1 + 0.01 * min(years, 20)) * rng.normal(1, 0.08), 2)

    dob = pd.Timestamp("1960-01-01") + pd.Timedelta(days=int(rng.integers(0, (pd.Timestamp("2003-12-31") - pd.Timestamp("1960-01-01")).days)))
    created = pd.Timestamp(ACCOUNTS_START) + pd.Timedelta(seconds=int(rng.integers(0, int((pd.Timestamp(ACCOUNTS_END) - pd.Timestamp(ACCOUNTS_START)).total_seconds()))))
    handy_users.append({
        "id": hid, "email": email, "password_hash": password_hash, "first_name": first, "last_name": last,
        "date_of_birth": dob.date(), "phone": phone,
        "house": house, "postal_code": a.postcode, "city": a.city, "state": a.state, "country": "Germany",
        "latitude": float(a.latitude), "longitude": float(a.longitude),
        "is_handyman": True, "created_at": created, "is_active": True,
    })
    details.append({
        "user_id": hid, "specialisations": specs, "skills": skills, "experience_summary": cv["summary"],
        "cv_path": f"{CV_DIR}/cv_{hid:06d}.pdf", "cv_raw_text": cv_to_text(cv),
        "has_car": "Own work van and tools" in cv["extras"],
        "completed_cases": 0, "rating_avg": 0.0, "rating_count": 0, "avg_price": price,
    })
    truth.append({"user_id": hid, "quality": quality, "years_experience": years, "primary_specialisation": specs[0], "cv_json": json.dumps(cv)})

handy_users = pd.DataFrame(handy_users)
details     = pd.DataFrame(details)
truth       = pd.DataFrame(truth)
assert handy_users["email"].is_unique and len(used_emails) == N_USERS + N_HANDYMEN and not handy_users.duplicated(subset=["house", "postal_code", "city"]).any()
print(details["specialisations"].map(lambda s: s[0]).value_counts())

specialisations
plumbing               1609
electrical             1526
appliance_repair       1236
painting               1033
heating_hvac            955
carpentry               934
general_maintenance     920
flooring                682
locksmith               579
roofing                 526
Name: count, dtype: int64


In [8]:
# Cell 4 — Save to Delta
from pyspark.sql import functions as F

# handyman accounts go into the same table as the clients written by 02_generate_users;
# replaceWhere swaps only the handyman rows, so re-running this notebook never duplicates them
(spark.createDataFrame(handy_users)
      .write.mode("overwrite").option("replaceWhere", "is_handyman = true")
      .saveAsTable(T_USERS))

(spark.createDataFrame(details)
      .withColumn("user_id", F.col("user_id").cast("bigint"))
      .withColumn("rating_avg", F.col("rating_avg").cast("decimal(3,2)"))
      .withColumn("avg_price",  F.col("avg_price").cast("decimal(10,2)"))
      .withColumn("completed_cases", F.col("completed_cases").cast("int"))
      .withColumn("rating_count",    F.col("rating_count").cast("int"))
      .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(T_HANDYMAN_DETAILS))

(spark.createDataFrame(truth)
      .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(T_HANDYMEN_TRUTH))

print(f"Saved {N_HANDYMEN:,} handyman users to {T_USERS} and their details to {T_HANDYMAN_DETAILS} (+ ground truth in {T_HANDYMEN_TRUTH})")
display(spark.table(T_HANDYMAN_DETAILS).drop("cv_raw_text").limit(10))

Saved 10,000 handymen to bootcamp_students.maintops.synth_handymen (+ ground truth in bootcamp_students.maintops.synth_handymen_truth)


,id,email,first_name,last_name,phone,house,postal_code,city,state,country,latitude,longitude,specialisations,skills,experience_summary,cv_path,completed_cases,rating_avg,rating_count,avg_price
0,7501,andrei_lorenz99@example.com,Andrei,Lorenz,+49 160 1733150,Koolbarg 40d,22117,Hamburg,Hamburg,Germany,53.546279,10.140870,"[appliance_repair, electrical, general_mainten...","[coffee machine descaling, extractor hood inst...",9 years of experience as appliance service tec...,/Volumes/bootcamp_students/maintops/maintops_d...,0,0.00,0,118.24
1,7502,mario-keller@example.com,Mario,Keller,+49 160 5590335,Fügener Weg 6,12209,Berlin,Berlin,Germany,52.415730,13.339885,[general_maintenance],"[minor repairs, flat-pack assembly, TV wall mo...",19 years of experience as property maintenance...,/Volumes/bootcamp_students/maintops/maintops_d...,0,0.00,0,126.56
2,7503,oliver-kaiser13@example.com,Oliver,Kaiser,+49 175 3685042,Weiherdamm 6,66459,Kirkel,Saarland,Germany,49.282619,7.221429,"[appliance_repair, electrical]","[extractor hood installation, dishwasher repai...",4 years of experience as appliance service tec...,/Volumes/bootcamp_students/maintops/maintops_d...,0,0.00,0,151.44
3,7504,mehmetschmitt17@example.com,Mehmet,Schmitt,+49 162 8375140,Amselstraße 12,66450,Bexbach,Saarland,Germany,49.348113,7.248368,"[plumbing, heating_hvac]","[tap replacement, pipe repair, drain unblockin...",4 years of experience as plumbing & sanitary t...,/Volumes/bootcamp_students/maintops/maintops_d...,0,0.00,0,211.30
4,7505,klaus_kramer@example.net,Klaus,Kramer,+49 157 2895121,Franz-Meyer-Straße 6,13587,Berlin,Berlin,Germany,52.556980,13.219201,[general_maintenance],"[curtain rail fitting, door hinge adjustment, ...",11 years of experience as property maintenance...,/Volumes/bootcamp_students/maintops/maintops_d...,0,0.00,0,133.46
5,7506,h.klein42@example.com,Holger,Klein,+49 175 3855645,Ludewigstraße 91,28277,Bremen,Bremen,Germany,53.030946,8.821342,[roofing],"[gutter installation, flat roof sealing, ridge...","8 years of experience as roofer, specialising ...",/Volumes/bootcamp_students/maintops/maintops_d...,0,0.00,0,660.33
6,7507,elif_busch@example.org,Elif,Busch,+49 175 2746020,Müssenredder 31,22399,Hamburg,Hamburg,Germany,53.655211,10.074234,[locksmith],"[burglary damage repair, key cutting, safe ope...","9 years of experience as locksmith, specialisi...",/Volumes/bootcamp_students/maintops/maintops_d...,0,0.00,0,140.43
7,7508,olga.jager10@example.org,Olga,Jager,+49 157 7979003,Kettinger Straße 8A,12305,Berlin,Berlin,Germany,52.416016,13.396948,[appliance_repair],"[freezer repair, oven repair, appliance instal...",19 years of experience as appliance service te...,/Volumes/bootcamp_students/maintops/maintops_d...,0,0.00,0,160.84
8,7509,yusuf-horn28@example.com,Yusuf,Horn,+49 151 4869481,Paul-Sorge-Straße 139a,22455,Hamburg,Hamburg,Germany,53.635040,9.951509,[carpentry],"[skirting boards, staircase repair, kitchen wo...","9 years of experience as carpenter, specialisi...",/Volumes/bootcamp_students/maintops/maintops_d...,0,0.00,0,280.07
9,7510,petrasommer57@example.net,Petra,Sommer,+49 175 1049898,Krippwiesstraße 5,66687,Wadern,Saarland,Germany,49.533448,6.894908,[locksmith],"[window lock fitting, lock opening, key cuttin...","3 years of experience as locksmith, specialisi...",/Volumes/bootcamp_students/maintops/maintops_d...,0,0.00,0,146.15
